# Advanced Problems with Solutions: Mapping, Starmapping, Reducing, and Accumulating

This notebook is a single, self-contained practice set for advanced use of:

- `map`
- `itertools.starmap`
- `functools.reduce`
- `itertools.accumulate`
- iterator laziness and exhaustion
- pure functional-style data pipelines
- prefix scans, event sourcing, MapReduce-style aggregation, and streaming analytics

Every problem includes a complete solution and executable tests.

## Best-practice checklist

Use this checklist while solving the problems:

1. Prefer **pure functions** in `map`, `starmap`, `reduce`, and `accumulate`.
2. Remember that `map`, `starmap`, `filter`, `zip`, and `accumulate` are **iterators**.
3. Do not consume an iterator twice unless you use `itertools.tee` or intentionally materialize it.
4. Use `operator.add`, `operator.mul`, `operator.itemgetter`, etc. when they make code clearer.
5. Use an explicit initializer for `reduce` when the input may be empty.
6. Avoid mutating an accumulator unless mutation is intentional and documented.
7. Keep pipelines readable: split complicated lambdas into named functions.
8. Test empty inputs, singleton inputs, repeated values, and edge cases.

## Setup

Run this cell once before the examples and problems.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import reduce
from itertools import accumulate, chain, groupby, islice, starmap, tee
from math import hypot
from operator import add, itemgetter, mul
from pprint import pprint
from typing import Any, Callable, Iterable, Iterator, TypeVar
import re
import timeit

T = TypeVar("T")
U = TypeVar("U")

_MISSING = object()


def take(n: int, iterable: Iterable[T]) -> list[T]:
    """Return the first n items from an iterable as a list."""
    return list(islice(iterable, n))


def assert_equal(actual: Any, expected: Any) -> str:
    """Tiny assertion helper for notebook-friendly tests."""
    assert actual == expected, f"\nactual:   {actual!r}\nexpected: {expected!r}"
    return "OK"


def almost_equal(actual: float, expected: float, *, tolerance: float = 1e-9) -> str:
    assert abs(actual - expected) <= tolerance, f"\nactual:   {actual!r}\nexpected: {expected!r}"
    return "OK"

## Example A: `map` is lazy and can be exhausted

A `map` object does not compute everything immediately. It computes values as you ask for them.

In [2]:
source = iter(range(10))
squares = map(lambda x: x * x, source)

first_three = take(3, squares)
next_three = take(3, squares)
rest = list(squares)
exhausted = list(squares)

assert_equal(first_three, [0, 1, 4])
assert_equal(next_three, [9, 16, 25])
assert_equal(rest, [36, 49, 64, 81])
assert_equal(exhausted, [])

'OK'

## Example B: `starmap` unpacks each tuple into positional arguments

Use `starmap(fn, rows)` when each row contains the positional arguments needed by `fn`.

In [3]:
points = [(0, 0), (3, 4), (-5, 12), (8, -15)]
distances = list(starmap(hypot, points))

assert_equal(distances, [0.0, 5.0, 13.0, 17.0])

'OK'

## Example C: `reduce` gives one final value; `accumulate` gives all intermediate values

In [4]:
values = [1, 2, 3, 4]

final_product = reduce(mul, values)
running_products = list(accumulate(values, mul))

assert_equal(final_product, 24)
assert_equal(running_products, [1, 2, 6, 24])

'OK'

---

# Problem 1 — Build a left-to-right function pipeline using `reduce`

Write `pipe(*funcs)` so that:

```python
pipe(f, g, h)(x) == h(g(f(x)))
```

Requirements:

- Use `reduce`.
- Support zero functions by returning the identity function.
- Keep the solution readable.

In [5]:
def pipe(*funcs: Callable[[Any], Any]) -> Callable[[Any], Any]:
    """Compose functions left-to-right."""

    def identity(x: Any) -> Any:
        return x

    def compose_left_to_right(f: Callable[[Any], Any], g: Callable[[Any], Any]) -> Callable[[Any], Any]:
        return lambda x: g(f(x))

    return reduce(compose_left_to_right, funcs, identity)


clean_text = pipe(
    str.strip,
    str.lower,
    lambda s: re.sub(r"\s+", " ", s),
    lambda s: s.replace("python", "Python"),
)

assert_equal(clean_text("   PYTHON     IS    FUN   "), "Python is fun")
assert_equal(pipe()(42), 42)

'OK'

---

# Problem 2 — Reimplement `reduce` safely

Write `reduce_left(fn, iterable, initializer)`.

Requirements:

- Match the core behavior of `functools.reduce`.
- If no initializer is provided and the iterable is empty, raise `TypeError`.
- Accept any iterable, not just lists.

In [6]:
def reduce_left(
    fn: Callable[[T, T], T],
    iterable: Iterable[T],
    initializer: Any = _MISSING,
) -> T:
    """A small educational implementation of functools.reduce."""
    iterator = iter(iterable)

    if initializer is _MISSING:
        try:
            accumulator = next(iterator)
        except StopIteration:
            raise TypeError("reduce_left() of empty iterable with no initial value")
    else:
        accumulator = initializer

    for item in iterator:
        accumulator = fn(accumulator, item)

    return accumulator


assert_equal(reduce_left(add, [10, 20, 30]), 60)
assert_equal(reduce_left(mul, [2, 3, 4], 10), 240)
assert_equal(reduce_left(add, [], 0), 0)

try:
    reduce_left(add, [])
except TypeError as ex:
    assert "empty iterable" in str(ex)
else:
    raise AssertionError("Expected TypeError")

---

# Problem 3 — Implement `accumulate` with an optional initializer

Older teaching material often shows `accumulate(iterable, fn)` without an initializer.
Modern Python supports `initial=...`, but implementing it yourself is a good iterator exercise.

Write `accumulate_with_initial(iterable, fn=add, initializer=_MISSING)`.

Expected behavior:

```python
list(accumulate_with_initial([1, 2, 3]))          # [1, 3, 6]
list(accumulate_with_initial([1, 2, 3], mul))     # [1, 2, 6]
list(accumulate_with_initial([1, 2, 3], add, 10)) # [10, 11, 13, 16]
```

In [7]:
def accumulate_with_initial(
    iterable: Iterable[T],
    fn: Callable[[Any, T], Any] = add,
    initializer: Any = _MISSING,
) -> Iterator[Any]:
    """Yield running accumulation values, optionally including an initializer."""
    iterator = iter(iterable)

    if initializer is _MISSING:
        try:
            accumulator = next(iterator)
        except StopIteration:
            return
        yield accumulator
    else:
        accumulator = initializer
        yield accumulator

    for item in iterator:
        accumulator = fn(accumulator, item)
        yield accumulator


assert_equal(list(accumulate_with_initial([1, 2, 3])), [1, 3, 6])
assert_equal(list(accumulate_with_initial([1, 2, 3], mul)), [1, 2, 6])
assert_equal(list(accumulate_with_initial([1, 2, 3], add, 10)), [10, 11, 13, 16])
assert_equal(list(accumulate_with_initial([], add, 0)), [0])
assert_equal(list(accumulate_with_initial([])), [])

'OK'

---

# Problem 4 — Use `starmap` for weighted distances

You receive rows in this form:

```python
(x, y, weight_x, weight_y)
```

For each row, compute:

```python
abs(x) * weight_x + abs(y) * weight_y
```

Requirements:

- Use `starmap`.
- Keep the calculation in a named function rather than a long lambda.

In [8]:
def weighted_manhattan(x: float, y: float, weight_x: float, weight_y: float) -> float:
    return abs(x) * weight_x + abs(y) * weight_y


rows = [
    (3, -4, 0.5, 2.0),
    (-10, 2, 1.0, 3.0),
    (0, -8, 100.0, 0.25),
]

scores = list(starmap(weighted_manhattan, rows))
assert_equal(scores, [9.5, 16.0, 2.0])

'OK'

---

# Problem 5 — Build a lazy sensor-cleaning pipeline

You receive raw sensor lines:

```text
DEVICE|TIMESTAMP|TEMPERATURE|STATUS
```

The temperature may be in Celsius or Fahrenheit, for example `21.5C` or `70.7F`.

Build a pipeline that:

1. Parses each line into a dictionary.
2. Converts Fahrenheit to Celsius.
3. Keeps only `ok` readings.
4. Keeps only physically reasonable Celsius values in `[-50, 80]`.
5. Does not materialize intermediate lists.

In [9]:
def parse_temperature(token: str) -> float:
    unit = token[-1].upper()
    value = float(token[:-1])

    if unit == "C":
        return value
    if unit == "F":
        return (value - 32) * 5 / 9

    raise ValueError(f"Unknown temperature unit: {token!r}")


def parse_reading(line: str) -> dict[str, Any]:
    device, timestamp, temperature, status = line.strip().split("|")
    return {
        "device": device,
        "timestamp": timestamp,
        "celsius": round(parse_temperature(temperature), 2),
        "status": status.lower(),
    }


def valid_reading(row: dict[str, Any]) -> bool:
    return row["status"] == "ok" and -50 <= row["celsius"] <= 80


def sensor_pipeline(lines: Iterable[str]) -> Iterator[dict[str, Any]]:
    parsed = map(parse_reading, lines)
    return filter(valid_reading, parsed)


raw_lines = [
    "A1|2026-07-06T10:00:00|21.5C|ok",
    "A1|2026-07-06T10:01:00|70.7F|ok",
    "A2|2026-07-06T10:02:00|999C|ok",
    "A3|2026-07-06T10:03:00|18C|bad",
]

clean_rows = list(sensor_pipeline(raw_lines))

assert_equal(
    clean_rows,
    [
        {"device": "A1", "timestamp": "2026-07-06T10:00:00", "celsius": 21.5, "status": "ok"},
        {"device": "A1", "timestamp": "2026-07-06T10:01:00", "celsius": 21.5, "status": "ok"},
    ],
)

'OK'

---

# Problem 6 — Running statistics with `accumulate`

Create running statistics for a stream of readings.

For each prefix of the input, track:

- count
- total
- minimum
- maximum
- mean

Use an immutable `dataclass` as the accumulator state.

In [10]:
@dataclass(frozen=True)
class RunningStats:
    count: int = 0
    total: float = 0.0
    minimum: float = float("inf")
    maximum: float = float("-inf")

    @property
    def mean(self) -> float | None:
        if self.count == 0:
            return None
        return self.total / self.count


def update_stats(stats: RunningStats, value: float) -> RunningStats:
    return RunningStats(
        count=stats.count + 1,
        total=stats.total + value,
        minimum=min(stats.minimum, value),
        maximum=max(stats.maximum, value),
    )


readings = [10.0, 20.0, 5.0, 30.0]
states = list(accumulate_with_initial(readings, update_stats, RunningStats()))
summary = [(s.count, s.total, s.minimum, s.maximum, s.mean) for s in states]

assert_equal(
    summary,
    [
        (0, 0.0, float("inf"), float("-inf"), None),
        (1, 10.0, 10.0, 10.0, 10.0),
        (2, 30.0, 10.0, 20.0, 15.0),
        (3, 35.0, 5.0, 20.0, 35.0 / 3),
        (4, 65.0, 5.0, 30.0, 16.25),
    ],
)

'OK'

---

# Problem 7 — Portfolio state from transactions

A trade is represented as:

```python
(side, quantity, price)
```

Where `side` is either `"BUY"` or `"SELL"`.

Build an audit trail of portfolio states after every trade.

Requirements:

- Use `accumulate_with_initial`.
- Do not mutate state objects.
- Track `shares` and `cash`.
- Buying decreases cash; selling increases cash.

In [11]:
@dataclass(frozen=True)
class Position:
    shares: int = 0
    cash: float = 0.0


def apply_trade(position: Position, trade: tuple[str, int, float]) -> Position:
    side, quantity, price = trade
    side = side.upper()

    if side == "BUY":
        share_delta = quantity
    elif side == "SELL":
        share_delta = -quantity
    else:
        raise ValueError(f"Unknown trade side: {side!r}")

    cash_delta = -share_delta * price
    return Position(
        shares=position.shares + share_delta,
        cash=round(position.cash + cash_delta, 2),
    )


trades = [
    ("BUY", 10, 100.00),
    ("BUY", 5, 110.00),
    ("SELL", 8, 120.00),
]

audit_trail = list(accumulate_with_initial(trades, apply_trade, Position()))

assert_equal(
    audit_trail,
    [
        Position(shares=0, cash=0.0),
        Position(shares=10, cash=-1000.0),
        Position(shares=15, cash=-1550.0),
        Position(shares=7, cash=-590.0),
    ],
)

'OK'

---

# Problem 8 — Cumulative returns and drawdowns

Given periodic returns like `0.02` for +2%, compute:

1. The equity curve starting at `1.0`.
2. The running peak of the equity curve.
3. The drawdown at every point.

Drawdown formula:

```python
equity / running_peak - 1
```

Use `accumulate`, `starmap`, and `mul`.

In [12]:
returns = [0.10, -0.05, 0.03, -0.20, 0.25]
growth_factors = map(lambda r: 1 + r, returns)

equity_curve = list(accumulate_with_initial(growth_factors, mul, 1.0))
running_peaks = list(accumulate(equity_curve, max))
drawdowns = list(starmap(lambda equity, peak: round(equity / peak - 1, 4), zip(equity_curve, running_peaks)))

assert_equal([round(x, 6) for x in equity_curve], [1.0, 1.1, 1.045, 1.07635, 0.86108, 1.07635])
assert_equal([round(x, 6) for x in running_peaks], [1.0, 1.1, 1.1, 1.1, 1.1, 1.1])
assert_equal(drawdowns, [0.0, 0.0, -0.05, -0.0215, -0.2172, -0.0215])

'OK'

---

# Problem 9 — Word count in MapReduce style

Build a word counter using the MapReduce idea:

1. Map each line to a `Counter`.
2. Reduce all counters into one final `Counter`.

Requirements:

- Ignore case.
- Extract only alphanumeric word tokens.
- Use `reduce` with an initializer.

In [13]:
WORD_RE = re.compile(r"[A-Za-z0-9]+")


def tokenize(text: str) -> list[str]:
    return [match.group(0).lower() for match in WORD_RE.finditer(text)]


def counter_for_line(line: str) -> Counter[str]:
    return Counter(tokenize(line))


def merge_counters(left: Counter[str], right: Counter[str]) -> Counter[str]:
    # Returning a new Counter keeps the reducer free from surprising side effects.
    return left + right


def word_count(lines: Iterable[str]) -> Counter[str]:
    return reduce(merge_counters, map(counter_for_line, lines), Counter())


lines = [
    "Map, map, reduce!",
    "Reduce and accumulate.",
    "MapReduce is not the same as Python reduce.",
]

counts = word_count(lines)
assert_equal(counts["map"], 2)
assert_equal(counts["reduce"], 3)
assert_equal(counts["accumulate"], 1)
assert_equal(counts["python"], 1)

'OK'

---

# Problem 10 — Build an inverted index with `starmap`, `chain`, and `groupby`

Given documents as `(doc_id, text)`, create an index:

```python
word -> sorted list of document IDs containing that word
```

Requirements:

- Use `starmap` to pass `(doc_id, text)` into a helper function.
- Use `chain.from_iterable` to flatten mapped pairs.
- Sort before `groupby`.

In [14]:
def word_doc_pairs(doc_id: str, text: str) -> list[tuple[str, str]]:
    return [(word, doc_id) for word in set(tokenize(text))]


def inverted_index(documents: Iterable[tuple[str, str]]) -> dict[str, list[str]]:
    pairs = chain.from_iterable(starmap(word_doc_pairs, documents))
    sorted_pairs = sorted(pairs, key=itemgetter(0, 1))

    index: dict[str, list[str]] = {}
    for word, group in groupby(sorted_pairs, key=itemgetter(0)):
        index[word] = sorted({doc_id for _, doc_id in group})
    return index


documents = [
    ("doc1", "Python map reduce accumulate"),
    ("doc2", "Python iterators are lazy"),
    ("doc3", "Reduce can power MapReduce style jobs"),
]

index = inverted_index(documents)
assert_equal(index["python"], ["doc1", "doc2"])
assert_equal(index["reduce"], ["doc1", "doc3"])
assert_equal(index["lazy"], ["doc2"])
assert_equal(index["mapreduce"], ["doc3"])

'OK'

---

# Problem 11 — Event sourcing with `reduce` and `accumulate`

Bank account events are represented as:

```python
(account_id, event_type, amount)
```

`event_type` is either `"deposit"` or `"withdraw"`.

Build:

1. `apply_event(state, event)`
2. `final_balances(events)` using `reduce`
3. `balance_audit(events)` using `accumulate_with_initial`

Requirements:

- Never mutate the input state.
- Reject withdrawals that would make an account negative.

In [15]:
def apply_event(state: dict[str, float], event: tuple[str, str, float]) -> dict[str, float]:
    account_id, event_type, amount = event
    if amount < 0:
        raise ValueError("Amount must be non-negative")

    current = state.get(account_id, 0.0)

    if event_type == "deposit":
        new_balance = current + amount
    elif event_type == "withdraw":
        new_balance = current - amount
    else:
        raise ValueError(f"Unknown event type: {event_type!r}")

    if new_balance < 0:
        raise ValueError(f"Insufficient funds for {account_id}")

    new_state = dict(state)
    new_state[account_id] = round(new_balance, 2)
    return new_state


def final_balances(events: Iterable[tuple[str, str, float]]) -> dict[str, float]:
    return reduce(apply_event, events, {})


def balance_audit(events: Iterable[tuple[str, str, float]]) -> list[dict[str, float]]:
    return list(accumulate_with_initial(events, apply_event, {}))


bank_events = [
    ("checking", "deposit", 100.00),
    ("savings", "deposit", 250.00),
    ("checking", "withdraw", 40.00),
    ("checking", "deposit", 10.00),
]

assert_equal(final_balances(bank_events), {"checking": 70.0, "savings": 250.0})
assert_equal(
    balance_audit(bank_events),
    [
        {},
        {"checking": 100.0},
        {"checking": 100.0, "savings": 250.0},
        {"checking": 60.0, "savings": 250.0},
        {"checking": 70.0, "savings": 250.0},
    ],
)

'OK'

---

# Problem 12 — Matrix-vector multiplication using `map` and `starmap`

Implement:

```python
dot(xs, ys)
matvec(matrix, vector)
```

Requirements:

- Use `starmap(mul, zip(xs, ys))` inside `dot`.
- Use `map` to apply `dot` to every matrix row.

In [16]:
def dot(xs: Iterable[float], ys: Iterable[float]) -> float:
    return sum(starmap(mul, zip(xs, ys)))


def matvec(matrix: Iterable[Iterable[float]], vector: Iterable[float]) -> list[float]:
    vector_tuple = tuple(vector)  # Reusable for every row; avoids consuming a one-shot iterator repeatedly.
    return list(map(lambda row: dot(row, vector_tuple), matrix))


matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]
vector = [10, 20, 30]

assert_equal(dot([1, 2, 3], [4, 5, 6]), 32)
assert_equal(matvec(matrix, vector), [140, 320, 500])

'OK'

---

# Problem 13 — Running best record with `accumulate`

Given records like:

```python
(name, score)
```

Return the best record seen so far after each input record.

Tie-breaking rule: keep the earlier record if scores are equal.

In [17]:
def better_score(best: tuple[str, int], candidate: tuple[str, int]) -> tuple[str, int]:
    if candidate[1] > best[1]:
        return candidate
    return best


def running_best(records: Iterable[tuple[str, int]]) -> list[tuple[str, int]]:
    return list(accumulate(records, better_score))


scores = [("Ana", 88), ("Bo", 91), ("Cy", 91), ("Dee", 95), ("Eli", 90)]
assert_equal(
    running_best(scores),
    [("Ana", 88), ("Bo", 91), ("Bo", 91), ("Dee", 95), ("Dee", 95)],
)

'OK'

---

# Problem 14 — Revenue aggregation from order lines

Each order line is:

```python
(sku, quantity, unit_price)
```

Build:

1. A list of `(sku, line_total)` using `starmap`.
2. A revenue dictionary using `reduce`.
3. A running revenue list using `accumulate_with_initial`.

Use non-mutating reducers.

In [18]:
def order_total(sku: str, quantity: int, unit_price: float) -> tuple[str, float]:
    return sku, round(quantity * unit_price, 2)


def add_revenue(accumulator: dict[str, float], item: tuple[str, float]) -> dict[str, float]:
    sku, line_total = item
    updated = dict(accumulator)
    updated[sku] = round(updated.get(sku, 0.0) + line_total, 2)
    return updated


orders = [
    ("pen", 10, 1.50),
    ("notebook", 3, 5.00),
    ("pen", 4, 1.50),
    ("bag", 1, 25.00),
]

line_totals = list(starmap(order_total, orders))
revenue = reduce(add_revenue, line_totals, {})
running_revenue = list(accumulate_with_initial(line_totals, add_revenue, {}))

assert_equal(line_totals, [("pen", 15.0), ("notebook", 15.0), ("pen", 6.0), ("bag", 25.0)])
assert_equal(revenue, {"pen": 21.0, "notebook": 15.0, "bag": 25.0})
assert_equal(
    running_revenue,
    [
        {},
        {"pen": 15.0},
        {"pen": 15.0, "notebook": 15.0},
        {"pen": 21.0, "notebook": 15.0},
        {"pen": 21.0, "notebook": 15.0, "bag": 25.0},
    ],
)

'OK'

---

# Problem 15 — Fix an iterator exhaustion bug

This implementation is broken:

```python
def broken_mean_and_count(values):
    total = sum(values)
    count = len(list(values))
    return total / count, count
```

The problem is that `sum(values)` may consume the iterator, leaving nothing for `list(values)`.

Write a correct single-pass implementation.

In [19]:
def mean_and_count(values: Iterable[float]) -> tuple[float | None, int]:
    total = 0.0
    count = 0

    for value in values:
        total += value
        count += 1

    if count == 0:
        return None, 0

    return total / count, count


assert_equal(mean_and_count(iter([10, 20, 30])), (20.0, 3))
assert_equal(mean_and_count([]), (None, 0))

'OK'

---

# Problem 16 — When two passes are really needed, use `tee` carefully

Sometimes you truly need two independent passes over a one-shot iterable.

Write `min_max_spread(values)` using `itertools.tee`.

Return:

```python
(minimum, maximum, maximum - minimum)
```

For an empty iterable, return `None`.

Note: `tee` can cache data internally, so use it carefully for huge streams.

In [20]:
def min_max_spread(values: Iterable[float]) -> tuple[float, float, float] | None:
    values_for_min, values_for_max = tee(values)

    try:
        minimum = min(values_for_min)
        maximum = max(values_for_max)
    except ValueError:
        return None

    return minimum, maximum, maximum - minimum


assert_equal(min_max_spread(iter([10, 4, 8, 20])), (4, 20, 16))
assert_equal(min_max_spread([]), None)

'OK'

---

# Problem 17 — Prefix checksums using `accumulate`

A simple rolling checksum can be built with this recurrence:

```python
next_checksum = (previous_checksum * 31 + byte) % 1_000_000_007
```

Write `prefix_checksums(data)` that returns the checksum after every byte.

Requirements:

- Convert characters to integer bytes with `map(ord, data)`.
- Use `accumulate_with_initial`.
- Do not include the initial zero in the returned result.

In [21]:
MOD = 1_000_000_007


def update_checksum(previous: int, byte: int) -> int:
    return (previous * 31 + byte) % MOD


def prefix_checksums(data: str) -> list[int]:
    checksums_with_initial = accumulate_with_initial(map(ord, data), update_checksum, 0)
    return list(islice(checksums_with_initial, 1, None))


assert_equal(prefix_checksums("abc"), [97, 3105, 96354])
assert_equal(prefix_checksums(""), [])

'OK'

---

# Problem 18 — Normalize rows using `starmap`

Rows arrive as:

```python
(name, earned_points, possible_points)
```

Return dictionaries with normalized percentages.

Requirements:

- Use `starmap`.
- Round percentages to 2 decimals.
- Treat zero possible points as `None`.

In [22]:
def normalize_score(name: str, earned: float, possible: float) -> dict[str, float | str | None]:
    percent = None if possible == 0 else round(earned / possible * 100, 2)
    return {"name": name, "earned": earned, "possible": possible, "percent": percent}


raw_scores = [
    ("Ana", 45, 50),
    ("Bo", 18, 20),
    ("Cy", 0, 0),
]

normalized = list(starmap(normalize_score, raw_scores))
assert_equal(
    normalized,
    [
        {"name": "Ana", "earned": 45, "possible": 50, "percent": 90.0},
        {"name": "Bo", "earned": 18, "possible": 20, "percent": 90.0},
        {"name": "Cy", "earned": 0, "possible": 0, "percent": None},
    ],
)

'OK'

---

# Problem 19 — Nested reduction: group totals by category

Each transaction is:

```python
(category, amount)
```

Build category totals using `reduce`.

Then convert totals into a sorted leaderboard from largest total to smallest.

In [23]:
def add_category_total(accumulator: dict[str, float], transaction: tuple[str, float]) -> dict[str, float]:
    category, amount = transaction
    updated = dict(accumulator)
    updated[category] = round(updated.get(category, 0.0) + amount, 2)
    return updated


def category_leaderboard(transactions: Iterable[tuple[str, float]]) -> list[tuple[str, float]]:
    totals = reduce(add_category_total, transactions, {})
    return sorted(totals.items(), key=lambda item: (-item[1], item[0]))


transactions = [
    ("books", 20.0),
    ("food", 12.5),
    ("books", 30.0),
    ("tools", 50.0),
    ("food", 37.5),
]

assert_equal(category_leaderboard(transactions), [("books", 50.0), ("food", 50.0), ("tools", 50.0)])
assert_equal(category_leaderboard([]), [])

'OK'

---

# Problem 20 — Advanced challenge: reusable mini MapReduce framework

Implement a tiny local MapReduce helper:

```python
map_reduce(records, mapper, reducer, initializer_factory)
```

Where:

- `mapper(record)` returns zero or more `(key, value)` pairs.
- Values are grouped by key.
- `reducer(accumulator, value)` merges one value into a key's accumulator.
- `initializer_factory()` returns a fresh accumulator for each key.

Then use it to build word positions:

```python
word -> [(doc_id, position), ...]
```

In [24]:
def map_reduce(
    records: Iterable[Any],
    mapper: Callable[[Any], Iterable[tuple[Any, Any]]],
    reducer: Callable[[Any, Any], Any],
    initializer_factory: Callable[[], Any],
) -> dict[Any, Any]:
    grouped: dict[Any, Any] = {}

    for key, value in chain.from_iterable(map(mapper, records)):
        if key not in grouped:
            grouped[key] = initializer_factory()
        grouped[key] = reducer(grouped[key], value)

    return grouped


def word_position_mapper(record: tuple[str, str]) -> list[tuple[str, tuple[str, int]]]:
    doc_id, text = record
    return [(word, (doc_id, position)) for position, word in enumerate(tokenize(text))]


def append_value(accumulator: list[Any], value: Any) -> list[Any]:
    # Non-mutating version for clarity and safer reasoning.
    return accumulator + [value]


position_index = map_reduce(
    documents,
    mapper=word_position_mapper,
    reducer=append_value,
    initializer_factory=list,
)

assert_equal(position_index["python"], [("doc1", 0), ("doc2", 0)])
assert_equal(position_index["reduce"], [("doc1", 2), ("doc3", 0)])
assert_equal(position_index["accumulate"], [("doc1", 3)])

'OK'

---

# Extra practice prompts

Use the solved problems above as templates. Try these variations:

1. Rewrite the sensor pipeline so invalid rows are captured as errors instead of raising exceptions.
2. Add transaction timestamps to the event-sourcing example and compute end-of-day balances.
3. Extend the mini MapReduce framework with a `combiner` step.
4. Implement `running_median` and compare why it is harder than running sum/product/min/max.
5. Use `starmap` to apply different functions to different argument tuples.
6. Compare memory usage between a fully materialized pipeline and a lazy pipeline.
7. Add property-style tests for `reduce_left` and `accumulate_with_initial`.

## Final review

You now have examples and solutions covering:

- Lazy iterator behavior
- `map` pipelines
- `starmap` unpacking
- `reduce` with and without initializers
- Custom `reduce` and custom `accumulate`
- Prefix scans and running state
- MapReduce-style word count and indexing
- Event sourcing
- Matrix-vector multiplication
- Iterator exhaustion bugs

A good next step is to re-run the notebook after deleting the solution code cells and solving each problem from scratch.